In [ ]:
import sys
sys.path.append('../../../code/libs/')

%load_ext autoreload
%autoreload 2
import utils
import viz
import ios
import constants
import text as txtlib

In [2]:
# import os
import pandas as pd
import numpy as np

In [3]:
ROOT = 'insurance_2015_2022/'
metric = 'insurance'

In [4]:
ios.get_files_from_pattern(ios.path_join(ROOT,"*acs_mod.xlsx"))

['insurance_2015_2022/hic04_acs_mod.xlsx']

In [5]:
import pandas as pd

# Path to the Excel file (replace with the actual file path)
file_path = ios.get_files_from_pattern(ios.path_join(ROOT,"*acs_mod.xlsx"))[0]

# Read the Excel file with all sheets
excel_data = pd.ExcelFile(file_path)

# Initialize a list to collect the data
consolidated_data = []

# Process each sheet (tab)
for sheet_name in excel_data.sheet_names:
    # Read the data from the current sheet
    tmp = excel_data.parse(sheet_name)
    
    # Identify the year from the sheet name
    try:
        year = int(sheet_name)
    except:
        continue
    
    tmp.drop(index=[0,1,3], inplace=True)
    tmp.reset_index(inplace=True, drop=True)
    tmp.columns = tmp.loc[0]  # Set the header row
    tmp.rename(columns={year:'Estimate', 'Nation/State':'state_name'}, inplace=True)
    tmp = tmp[1:]
    tmp.drop(columns=[c for c in tmp.columns if c not in ['state_name','Coverage','Estimate']], inplace=True)
    tmp.loc[:,'year'] = year
    tmp.reset_index(inplace=True, drop=True)
    tmp.columns.name = None
    
    data = None
    for coverage in ['Any coverage', 'Uninsured', 'Private', 'Public']:
        tmp2 = tmp.query("Coverage == @coverage").copy()
        tmp2.set_index(["state_name",'year'], inplace=True)
        tmp2.drop(columns=['Coverage'], inplace=True)
        colname = f"{metric}_{coverage.replace(' ','_').lower()}"
        tmp2.rename(columns={'Estimate':colname}, inplace=True)
        
        data = data.join(tmp2) if data is not None else tmp2.copy()

    consolidated_data.append(data)

consolidated_data = pd.concat(consolidated_data, ignore_index=False)
consolidated_data.shape

(408, 4)

In [6]:
consolidated_data.head(10)

,,insurance_any_coverage,insurance_uninsured,insurance_private,insurance_public
state_name,year,,,,
Alabama,2015,4297,484,3198,1752
Alaska,2015,607,106,478,205
Arizona,2015,5991,728,4222,2586
Arkansas,2015,2647,278,1793,1231
California,2015,35330,3317,24140,14670
Colorado,2015,4935,433,3778,1719
Connecticut,2015,3327,211,2545,1185
Delaware,2015,877,54,664,348
District of Columbia,2015,636,25,466,231


In [7]:
ios.save_csv(consolidated_data, 'insurance_2015_2022.csv')